In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

# pyspark.sql.functions.from_json

<img src="images/from_json.jpg" align = "left">



Ref - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_json.html

In PySpark, `from_json` is a powerful function used to transform a string column containing JSON data into a structured format (like a `StructType` or a `MapType`). This is essential when dealing with data from Kafka, NoSQL databases, or raw text files where nested data is packed into a single string.

# 1. How it Works

The function requires two primary arguments:

**The Column:** The string column containing the JSON.

**The Schema:** A definition of what the JSON looks like so Spark knows how to parse it.

## The Logic Flow

**1. Read:** Spark looks at the string.

**2. Schema Mapping:** It attempts to map the JSON keys to the fields defined in your schema.

**Output:** It returns a new column of type `Struct`, which you can then "flatten" to get individual columns.

# 2. Working Example

Let's say you have a DataFrame where one column is a JSON string representing a user's profile.

**Step 1:** Define the Schema

First, we use `StructType` to define the expected structure of the JSON.

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import from_json, col

# Define the schema of the JSON inside the string
json_schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True)
])

**Step 2:** Create the Raw Data

In [5]:
data = [
    (1, '{"name": "Alice", "age": 30, "city": "New York"}'),
    (2, '{"name": "Bob", "age": 25, "city": "Seattle"}')
]
df = spark.createDataFrame(data, ["id", "raw_json"])
df.show(truncate = False)

+---+------------------------------------------------+
|id |raw_json                                        |
+---+------------------------------------------------+
|1  |{"name": "Alice", "age": 30, "city": "New York"}|
|2  |{"name": "Bob", "age": 25, "city": "Seattle"}   |
+---+------------------------------------------------+



**Step 3:** Apply `from_json`

We apply the function and then use `.*` to expand the struct into individual columns.

In [6]:
# Convert the string to a Struct
parsed_df = df.withColumn("data", from_json(col("raw_json"), json_schema))
print(parsed_df.printSchema())

# Flatten the struct to see individual columns
final_df = parsed_df.select("id", "data.*")

final_df.show()

root
 |-- id: long (nullable = true)
 |-- raw_json: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)
 |    |-- city: string (nullable = true)

None
+---+-----+---+--------+
| id| name|age|    city|
+---+-----+---+--------+
|  1|Alice| 30|New York|
|  2|  Bob| 25| Seattle|
+---+-----+---+--------+



# 3. Important Considerations

- **Corrupt Records:** If the JSON string doesn't match the schema (e.g., a letter where an integer is expected), PySpark will return null for those fields by default.

- **Options:** You can pass a dictionary of options as a third argument. For example, {"allowBackslashEscapingAnyCharacter": "true"}.

- **Performance:** Parsing JSON is computationally expensive. If you are doing this at scale (billions of rows), ensure your schema is as tight as possible (avoid unnecessary fields).